### **Key Considerations for Your Dataproc Cluster**

1. **Cluster Resources:**

    - **Master:** `n2-standard-4` (4 vCPUs, 16 GB RAM, 44GB disk)
    - **Workers (2x):** `n2-standard-4` (4 vCPUs, 16 GB RAM, 44GB disk each)
    - **Total:** 8 worker vCPUs, ~32 GB RAM (excluding master node)

2. **Dataproc Features Disabled:**

    - No **autoscaling**, **Metastore**, **advanced execution layer**, **advanced optimizations**
    - **Storage:** `pd-balanced` (no SSDs, so I/O optimization is crucial)
    - **Networking:** Internal IP **enabled**

3. **Optimization Strategy:**

    - Tune **shuffle partitions**, **broadcast join threshold**, and **storage persistence**
    - Adjust **parallelism** based on **2 workers x 4 cores**
    - Avoid **excessive caching** due to **disk-based storage**


In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark=SparkSession.builder\
.appName('Olist Ecommerce Optimization')\
.config('spark.executor.memory','6g')\
.config('spark.executor.cores','4')\
.config('spark.driver.memory','4g')\
.config('spark.MaxResultSize','2g')\
.config('spark.shuffle.partitions','64')\
.config('spark.default.parallelism','64')\
.config('spark.sql.adaptive.enabled','true')\
.config('spark.sql.adaptive.coalescePartition.enabled','true')\
.config('autoBroadcastJoinThreshold',20*1024*1024)\
.config('spark.sql.files.maxPartitionBytes','64MB')\
.config('spark.sql.files.openCostInBytes','2MB')\
.config('spark.memory.storagFraction',0.2)\
.config('spark.memory.fraction',0.8)\
.getOrCreate()

In [ ]:
hdfs_path='/data/olist/'

In [ ]:
customers_df=spark.read.csv(hdfs_path + 'olist_customers_dataset.csv',header=True,inferSchema=True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv',header=True,inferSchema=True)
order_item_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv',header=True,inferSchema=True)
payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv',header=True,inferSchema=True)
reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv',header=True,inferSchema=True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv',header=True,inferSchema=True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header=True,inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header=True,inferSchema=True)
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True,inferSchema=True)


In [ ]:
full_orders_df=spark.read.parquet('/data/olist/proc')

# Optimized Join Strategies

In [ ]:
customer_broadcast_df=broadcast(customers_df)
optimized_broadcast_join=full_orders_df.join(customer_broadcast_df,'customer_id')

In [ ]:
# sort and merge join
sorted_customers_df=customers_df.sortWithinPartitions('customer_id')
sorted_orders_df=full_orders_df.sortWithinPartitions('customer_id')

In [ ]:
optimize_full_orders_merge_df=sorted_orders_df.join(sorted_customers_df,'customer_id')

In [ ]:
# Bucket Join
bucketed_customers_df=customers_df.repartition(10,'customer_id')
bucketed_orders_df=full_orders_df.repartition(10,'customer_id')
optimize_bucketed_fullorders_df=bucketed_orders_df.join(bucketed_customers_df,'customer_id')